# Return Period Analysis Storm Hans

## 0.5° Resolution/ERA5 Analysis/2day Precip.: Time-Series of Weighted Catchment Daily Precipitation and Return Period Analysis

### Dataset/resolution choices

In [1]:
# %% [Setup & dataset selection]
# ─────────────────────────────────────────────────────────────────────────────
# CHANGE ONLY THE TWO LINES MARKED ← to switch dataset or accumulation window.
# DATASET, RESOLUTION, and WEIGHT_DIR are derived automatically.
# ─────────────────────────────────────────────────────────────────────────────

import sys
from pathlib import Path

HELPER_DIR = Path("/nird/home/lbal/internship_storm_hans/helper")
if str(HELPER_DIR) not in sys.path:
    sys.path.insert(0, str(HELPER_DIR))

import config_paths as cfg

# ── Dataset selection ─────────────────────────────────────────────────────────
# Options:  "era5_0.5"  |  "era5_0.25"  |  "senorge"
DATASET_KEY = "era5_0.5"      # ← CHANGE THIS

# ── Accumulation window ───────────────────────────────────────────────────────
WINDOW_DAYS = 2              # ← CHANGE THIS (1 or 2)

# ── Recompute flag ────────────────────────────────────────────────────────────
FORCE_RECOMPUTE = False

# ── Figure subfolder ─────────────────────────────────────────────────────────
FIG_SUBDIR = "timeseries_return_hans"

# ── Derived automatically — do not edit below ────────────────────────────────
_CONFIG = {
    "era5_0.5":  {"dataset": "era5",    "resolution": "0.5x0.5",   "weight_dir": None},
    "era5_0.25": {"dataset": "era5",    "resolution": "0.25x0.25", "weight_dir": cfg.WEIGHTS_025_DIR},
    "senorge":   {"dataset": "senorge", "resolution": "",           "weight_dir": None},
}
if DATASET_KEY not in _CONFIG:
    raise ValueError(f"Unknown DATASET_KEY '{DATASET_KEY}'. Choose from: {list(_CONFIG)}")

DATASET    = _CONFIG[DATASET_KEY]["dataset"]
RESOLUTION = _CONFIG[DATASET_KEY]["resolution"]
WEIGHT_DIR = _CONFIG[DATASET_KEY]["weight_dir"]

print(f"Dataset    : {DATASET}")
print(f"Resolution : {RESOLUTION or 'n/a (seNorge — one grid only)'}")
print(f"Window     : {WINDOW_DAYS}-day")
print(f"Recompute  : {FORCE_RECOMPUTE}")
print(f"Weight dir : {WEIGHT_DIR or '(default: CATCHMENT_RAW_DIR)'}")
print(f"Fig subdir : {FIG_SUBDIR}")

Dataset    : era5
Resolution : 0.5x0.5
Window     : 2-day
Recompute  : False
Weight dir : (default: CATCHMENT_RAW_DIR)
Fig subdir : timeseries_return_hans


### Configuration check

Verify that all paths resolve correctly before running the full analysis

Verify if data is already stored in post-processed version or needs to be calculated with raw dataset

In [2]:
# %% [Configuration check]
import config_paths as cfg

tag = cfg.res_tag(DATASET, RESOLUTION)

print(f"Resolution tag     : {tag}")
print(f"Postproc dir       : {cfg.postproc_dir(DATASET)}")
print(f"Figures (primary)  : {cfg.FIGURES_DIR / FIG_SUBDIR}")
print(f"Figures (secondary): {cfg.FIGURES_DIR_SECONDARY / FIG_SUBDIR}")
print(f"Hans year          : {cfg.HANS_SEARCH_YEAR}")

# ── File discovery — branches on dataset ─────────────────────────────────────
if DATASET == "senorge":
    from data_senorge import find_senorge_files, get_year_range_senorge
    _raw_files = find_senorge_files(cfg.SENORGE_RAW_DIR)
    start_year, end_year = get_year_range_senorge(_raw_files)
    print(f"seNorge raw dir    : {cfg.SENORGE_RAW_DIR}")
else:
    from data_era5 import find_era5_files, get_year_range
    _raw_files = find_era5_files(cfg.ERA5_RAW_DIR, RESOLUTION)
    start_year, end_year = get_year_range(_raw_files, RESOLUTION)
    print(f"ERA5 raw dir       : {cfg.ERA5_RAW_DIR}")

print(f"Files found        : {len(_raw_files)}  ({start_year}–{end_year})")
print()
print("Postprocessed cache status:")
cached_count = 0
missing = []

for slug, title in cfg.CATCHMENTS.items():
    nc = cfg.catchment_postproc_path(
        DATASET, RESOLUTION, WINDOW_DAYS, slug, start_year, end_year
    )
    exists = nc.exists()
    if exists:
        cached_count += 1
    else:
        missing.append(slug)
    print(f"  {slug:30s}  {'✓ cached' if exists else '  (not yet cached)'}")

print()
print(f"Cached files       : {cached_count}/{len(cfg.CATCHMENTS)}")

if FORCE_RECOMPUTE:
    print("Run behavior       : FORCE_RECOMPUTE=True → all catchments will be recomputed from raw data")
elif missing:
    print("Run behavior       : cached catchments will be reused; missing catchments computed from raw data")
else:
    print("Run behavior       : all catchments cached → raw data will not be loaded")

Resolution tag     : era5_0.5x0.5
Postproc dir       : /nird/datalake/NS9873K/lbal/postprocessed/era5
Figures (primary)  : /nird/datalake/NS9873K/lbal/figures/timeseries_return_hans
Figures (secondary): /nird/home/lbal/internship_storm_hans/figures/timeseries_return_hans
Hans year          : 2023
ERA5 raw dir       : /nird/datapeak/NS9873K/etdu/raw/era5/continuous-format/europe/daily/tp24
Files found        : 84  (1941–2024)

Postprocessed cache status:
  nevina_bergheim                 ✓ cached
  nevina_honnefoss                ✓ cached
  nevina_losna                    ✓ cached
  regine_drammen                  ✓ cached
  regine_glomma                   ✓ cached

Cached files       : 5/5
Run behavior       : all catchments cached → raw data will not be loaded


### Run full analysis on all catchments

Loops over all five catchments automatically.  
For each catchment:
1. Loads or computes the weighted daily time series (cached to `postprocessed/`)
2. Fits GEV to annual maxima
3. Saves a two-panel PDF to `figures/timeseries_return_hans/`

**First run:** `FORCE_RECOMPUTE = False` still works — cache is missing so it computes automatically.  
**Subsequent runs:** cached NetCDFs are reused, only the figures are regenerated.

In [3]:
# %% [Run full analysis]
from catchment_tools import run_all

run_all(
    dataset         = DATASET,
    resolution      = RESOLUTION,
    window_days     = WINDOW_DAYS,
    force_recompute = FORCE_RECOMPUTE,
    fig_subdir      = FIG_SUBDIR,
    weight_dir      = WEIGHT_DIR,
)


[run_all] Dataset: era5 | Resolution: 0.5x0.5
[run_all] Files found: 84  (1941–2024)

── Catchment: Nevina Bergheim (nevina_bergheim) ──
  [cache] Found postprocessed file → post_processed_era5_0.5x0.5_2day_nevina_bergheim_1941-2024.nc
    [fig]   Saved → /nird/datalake/NS9873K/lbal/figures/timeseries_return_hans/timeseries_returnperiod_hans_era5_0.5x0.5_2day_nevina_bergheim_1941-2024.pdf
    [fig]   Saved → /nird/home/lbal/internship_storm_hans/figures/timeseries_return_hans/timeseries_returnperiod_hans_era5_0.5x0.5_2day_nevina_bergheim_1941-2024.pdf

── Catchment: Nevina Hønnefoss (nevina_honnefoss) ──
  [cache] Found postprocessed file → post_processed_era5_0.5x0.5_2day_nevina_honnefoss_1941-2024.nc
    [fig]   Saved → /nird/datalake/NS9873K/lbal/figures/timeseries_return_hans/timeseries_returnperiod_hans_era5_0.5x0.5_2day_nevina_honnefoss_1941-2024.pdf
    [fig]   Saved → /nird/home/lbal/internship_storm_hans/figures/timeseries_return_hans/timeseries_returnperiod_hans_era5_0.5x0.5

## 0.5° Resolution/ERA5 Analysis/1-day Precip.: Time-Series of Weighted Catchment Daily Precipitation and Return Period Analysis

### Dataset/resolution choices

In [4]:
# %% [Setup & dataset selection]
# ─────────────────────────────────────────────────────────────────────────────
# CHANGE ONLY THE TWO LINES MARKED ← to switch dataset or accumulation window.
# DATASET, RESOLUTION, and WEIGHT_DIR are derived automatically.
# ─────────────────────────────────────────────────────────────────────────────

import sys
from pathlib import Path

HELPER_DIR = Path("/nird/home/lbal/internship_storm_hans/helper")
if str(HELPER_DIR) not in sys.path:
    sys.path.insert(0, str(HELPER_DIR))

import config_paths as cfg

# ── Dataset selection ─────────────────────────────────────────────────────────
# Options:  "era5_0.5"  |  "era5_0.25"  |  "senorge"
DATASET_KEY = "era5_0.5"      # ← CHANGE THIS

# ── Accumulation window ───────────────────────────────────────────────────────
WINDOW_DAYS = 1              # ← CHANGE THIS (1 or 2)

# ── Recompute flag ────────────────────────────────────────────────────────────
FORCE_RECOMPUTE = False

# ── Figure subfolder ─────────────────────────────────────────────────────────
FIG_SUBDIR = "timeseries_return_hans"

# ── Derived automatically — do not edit below ────────────────────────────────
_CONFIG = {
    "era5_0.5":  {"dataset": "era5",    "resolution": "0.5x0.5",   "weight_dir": None},
    "era5_0.25": {"dataset": "era5",    "resolution": "0.25x0.25", "weight_dir": cfg.WEIGHTS_025_DIR},
    "senorge":   {"dataset": "senorge", "resolution": "",           "weight_dir": None},
}
if DATASET_KEY not in _CONFIG:
    raise ValueError(f"Unknown DATASET_KEY '{DATASET_KEY}'. Choose from: {list(_CONFIG)}")

DATASET    = _CONFIG[DATASET_KEY]["dataset"]
RESOLUTION = _CONFIG[DATASET_KEY]["resolution"]
WEIGHT_DIR = _CONFIG[DATASET_KEY]["weight_dir"]

print(f"Dataset    : {DATASET}")
print(f"Resolution : {RESOLUTION or 'n/a (seNorge — one grid only)'}")
print(f"Window     : {WINDOW_DAYS}-day")
print(f"Recompute  : {FORCE_RECOMPUTE}")
print(f"Weight dir : {WEIGHT_DIR or '(default: CATCHMENT_RAW_DIR)'}")
print(f"Fig subdir : {FIG_SUBDIR}")

Dataset    : era5
Resolution : 0.5x0.5
Window     : 1-day
Recompute  : False
Weight dir : (default: CATCHMENT_RAW_DIR)
Fig subdir : timeseries_return_hans


### Configuration check

In [5]:
# %% [Configuration check]
import config_paths as cfg

tag = cfg.res_tag(DATASET, RESOLUTION)

print(f"Resolution tag     : {tag}")
print(f"Postproc dir       : {cfg.postproc_dir(DATASET)}")
print(f"Figures (primary)  : {cfg.FIGURES_DIR / FIG_SUBDIR}")
print(f"Figures (secondary): {cfg.FIGURES_DIR_SECONDARY / FIG_SUBDIR}")
print(f"Hans year          : {cfg.HANS_SEARCH_YEAR}")

# ── File discovery — branches on dataset ─────────────────────────────────────
if DATASET == "senorge":
    from data_senorge import find_senorge_files, get_year_range_senorge
    _raw_files = find_senorge_files(cfg.SENORGE_RAW_DIR)
    start_year, end_year = get_year_range_senorge(_raw_files)
    print(f"seNorge raw dir    : {cfg.SENORGE_RAW_DIR}")
else:
    from data_era5 import find_era5_files, get_year_range
    _raw_files = find_era5_files(cfg.ERA5_RAW_DIR, RESOLUTION)
    start_year, end_year = get_year_range(_raw_files, RESOLUTION)
    print(f"ERA5 raw dir       : {cfg.ERA5_RAW_DIR}")

print(f"Files found        : {len(_raw_files)}  ({start_year}–{end_year})")
print()
print("Postprocessed cache status:")
cached_count = 0
missing = []

for slug, title in cfg.CATCHMENTS.items():
    nc = cfg.catchment_postproc_path(
        DATASET, RESOLUTION, WINDOW_DAYS, slug, start_year, end_year
    )
    exists = nc.exists()
    if exists:
        cached_count += 1
    else:
        missing.append(slug)
    print(f"  {slug:30s}  {'✓ cached' if exists else '  (not yet cached)'}")

print()
print(f"Cached files       : {cached_count}/{len(cfg.CATCHMENTS)}")

if FORCE_RECOMPUTE:
    print("Run behavior       : FORCE_RECOMPUTE=True → all catchments will be recomputed from raw data")
elif missing:
    print("Run behavior       : cached catchments will be reused; missing catchments computed from raw data")
else:
    print("Run behavior       : all catchments cached → raw data will not be loaded")

Resolution tag     : era5_0.5x0.5
Postproc dir       : /nird/datalake/NS9873K/lbal/postprocessed/era5
Figures (primary)  : /nird/datalake/NS9873K/lbal/figures/timeseries_return_hans
Figures (secondary): /nird/home/lbal/internship_storm_hans/figures/timeseries_return_hans
Hans year          : 2023
ERA5 raw dir       : /nird/datapeak/NS9873K/etdu/raw/era5/continuous-format/europe/daily/tp24
Files found        : 84  (1941–2024)

Postprocessed cache status:
  nevina_bergheim                 ✓ cached
  nevina_honnefoss                ✓ cached
  nevina_losna                    ✓ cached
  regine_drammen                  ✓ cached
  regine_glomma                   ✓ cached

Cached files       : 5/5
Run behavior       : all catchments cached → raw data will not be loaded


### Run full analysis on all catchments

In [6]:
# %% [Run full analysis]
from catchment_tools import run_all

run_all(
    dataset         = DATASET,
    resolution      = RESOLUTION,
    window_days     = WINDOW_DAYS,
    force_recompute = FORCE_RECOMPUTE,
    fig_subdir      = FIG_SUBDIR,
    weight_dir      = WEIGHT_DIR,
)


[run_all] Dataset: era5 | Resolution: 0.5x0.5
[run_all] Files found: 84  (1941–2024)

── Catchment: Nevina Bergheim (nevina_bergheim) ──
  [cache] Found postprocessed file → post_processed_era5_0.5x0.5_1day_nevina_bergheim_1941-2024.nc
    [fig]   Saved → /nird/datalake/NS9873K/lbal/figures/timeseries_return_hans/timeseries_returnperiod_hans_era5_0.5x0.5_1day_nevina_bergheim_1941-2024.pdf
    [fig]   Saved → /nird/home/lbal/internship_storm_hans/figures/timeseries_return_hans/timeseries_returnperiod_hans_era5_0.5x0.5_1day_nevina_bergheim_1941-2024.pdf

── Catchment: Nevina Hønnefoss (nevina_honnefoss) ──
  [cache] Found postprocessed file → post_processed_era5_0.5x0.5_1day_nevina_honnefoss_1941-2024.nc
    [fig]   Saved → /nird/datalake/NS9873K/lbal/figures/timeseries_return_hans/timeseries_returnperiod_hans_era5_0.5x0.5_1day_nevina_honnefoss_1941-2024.pdf
    [fig]   Saved → /nird/home/lbal/internship_storm_hans/figures/timeseries_return_hans/timeseries_returnperiod_hans_era5_0.5x0.5

## 0.25° Resolution/ERA5 Analysis/2-day Precip.: Time-Series of Weighted Catchment Daily Precipitation and Return Period Analysis

### Dataset/resolution choices

In [7]:
# %% [Setup & dataset selection]
# ─────────────────────────────────────────────────────────────────────────────
# CHANGE ONLY THE TWO LINES MARKED ← to switch dataset or accumulation window.
# DATASET, RESOLUTION, and WEIGHT_DIR are derived automatically.
# ─────────────────────────────────────────────────────────────────────────────

import sys
from pathlib import Path

HELPER_DIR = Path("/nird/home/lbal/internship_storm_hans/helper")
if str(HELPER_DIR) not in sys.path:
    sys.path.insert(0, str(HELPER_DIR))

import config_paths as cfg

# ── Dataset selection ─────────────────────────────────────────────────────────
# Options:  "era5_0.5"  |  "era5_0.25"  |  "senorge"
DATASET_KEY = "era5_0.25"      # ← CHANGE THIS

# ── Accumulation window ───────────────────────────────────────────────────────
WINDOW_DAYS = 2              # ← CHANGE THIS (1 or 2)

# ── Recompute flag ────────────────────────────────────────────────────────────
FORCE_RECOMPUTE = False

# ── Figure subfolder ─────────────────────────────────────────────────────────
FIG_SUBDIR = "timeseries_return_hans"

# ── Derived automatically — do not edit below ────────────────────────────────
_CONFIG = {
    "era5_0.5":  {"dataset": "era5",    "resolution": "0.5x0.5",   "weight_dir": None},
    "era5_0.25": {"dataset": "era5",    "resolution": "0.25x0.25", "weight_dir": cfg.WEIGHTS_025_DIR},
    "senorge":   {"dataset": "senorge", "resolution": "",           "weight_dir": None},
}
if DATASET_KEY not in _CONFIG:
    raise ValueError(f"Unknown DATASET_KEY '{DATASET_KEY}'. Choose from: {list(_CONFIG)}")

DATASET    = _CONFIG[DATASET_KEY]["dataset"]
RESOLUTION = _CONFIG[DATASET_KEY]["resolution"]
WEIGHT_DIR = _CONFIG[DATASET_KEY]["weight_dir"]

print(f"Dataset    : {DATASET}")
print(f"Resolution : {RESOLUTION or 'n/a (seNorge — one grid only)'}")
print(f"Window     : {WINDOW_DAYS}-day")
print(f"Recompute  : {FORCE_RECOMPUTE}")
print(f"Weight dir : {WEIGHT_DIR or '(default: CATCHMENT_RAW_DIR)'}")
print(f"Fig subdir : {FIG_SUBDIR}")

Dataset    : era5
Resolution : 0.25x0.25
Window     : 2-day
Recompute  : False
Weight dir : /nird/datalake/NS9873K/lbal/postprocessed/0.25_weights
Fig subdir : timeseries_return_hans


### Configuration check

In [8]:
# %% [Configuration check]
import config_paths as cfg

tag = cfg.res_tag(DATASET, RESOLUTION)

print(f"Resolution tag     : {tag}")
print(f"Postproc dir       : {cfg.postproc_dir(DATASET)}")
print(f"Figures (primary)  : {cfg.FIGURES_DIR / FIG_SUBDIR}")
print(f"Figures (secondary): {cfg.FIGURES_DIR_SECONDARY / FIG_SUBDIR}")
print(f"Hans year          : {cfg.HANS_SEARCH_YEAR}")

# ── File discovery — branches on dataset ─────────────────────────────────────
if DATASET == "senorge":
    from data_senorge import find_senorge_files, get_year_range_senorge
    _raw_files = find_senorge_files(cfg.SENORGE_RAW_DIR)
    start_year, end_year = get_year_range_senorge(_raw_files)
    print(f"seNorge raw dir    : {cfg.SENORGE_RAW_DIR}")
else:
    from data_era5 import find_era5_files, get_year_range
    _raw_files = find_era5_files(cfg.ERA5_RAW_DIR, RESOLUTION)
    start_year, end_year = get_year_range(_raw_files, RESOLUTION)
    print(f"ERA5 raw dir       : {cfg.ERA5_RAW_DIR}")

print(f"Files found        : {len(_raw_files)}  ({start_year}–{end_year})")
print()
print("Postprocessed cache status:")
cached_count = 0
missing = []

for slug, title in cfg.CATCHMENTS.items():
    nc = cfg.catchment_postproc_path(
        DATASET, RESOLUTION, WINDOW_DAYS, slug, start_year, end_year
    )
    exists = nc.exists()
    if exists:
        cached_count += 1
    else:
        missing.append(slug)
    print(f"  {slug:30s}  {'✓ cached' if exists else '  (not yet cached)'}")

print()
print(f"Cached files       : {cached_count}/{len(cfg.CATCHMENTS)}")

if FORCE_RECOMPUTE:
    print("Run behavior       : FORCE_RECOMPUTE=True → all catchments will be recomputed from raw data")
elif missing:
    print("Run behavior       : cached catchments will be reused; missing catchments computed from raw data")
else:
    print("Run behavior       : all catchments cached → raw data will not be loaded")

Resolution tag     : era5_0.25x0.25
Postproc dir       : /nird/datalake/NS9873K/lbal/postprocessed/era5
Figures (primary)  : /nird/datalake/NS9873K/lbal/figures/timeseries_return_hans
Figures (secondary): /nird/home/lbal/internship_storm_hans/figures/timeseries_return_hans
Hans year          : 2023
ERA5 raw dir       : /nird/datapeak/NS9873K/etdu/raw/era5/continuous-format/europe/daily/tp24
Files found        : 84  (1941–2024)

Postprocessed cache status:
  nevina_bergheim                 ✓ cached
  nevina_honnefoss                ✓ cached
  nevina_losna                    ✓ cached
  regine_drammen                  ✓ cached
  regine_glomma                   ✓ cached

Cached files       : 5/5
Run behavior       : all catchments cached → raw data will not be loaded


### Run full analysis on all catchments

In [9]:
# %% [Run full analysis]
from catchment_tools import run_all

run_all(
    dataset         = DATASET,
    resolution      = RESOLUTION,
    window_days     = WINDOW_DAYS,
    force_recompute = FORCE_RECOMPUTE,
    fig_subdir      = FIG_SUBDIR,
    weight_dir      = WEIGHT_DIR,
)


[run_all] Dataset: era5 | Resolution: 0.25x0.25
[run_all] Files found: 84  (1941–2024)

── Catchment: Nevina Bergheim (nevina_bergheim) ──
  [cache] Found postprocessed file → post_processed_era5_0.25x0.25_2day_nevina_bergheim_1941-2024.nc
    [fig]   Saved → /nird/datalake/NS9873K/lbal/figures/timeseries_return_hans/timeseries_returnperiod_hans_era5_0.25x0.25_2day_nevina_bergheim_1941-2024.pdf
    [fig]   Saved → /nird/home/lbal/internship_storm_hans/figures/timeseries_return_hans/timeseries_returnperiod_hans_era5_0.25x0.25_2day_nevina_bergheim_1941-2024.pdf

── Catchment: Nevina Hønnefoss (nevina_honnefoss) ──
  [cache] Found postprocessed file → post_processed_era5_0.25x0.25_2day_nevina_honnefoss_1941-2024.nc
    [fig]   Saved → /nird/datalake/NS9873K/lbal/figures/timeseries_return_hans/timeseries_returnperiod_hans_era5_0.25x0.25_2day_nevina_honnefoss_1941-2024.pdf
    [fig]   Saved → /nird/home/lbal/internship_storm_hans/figures/timeseries_return_hans/timeseries_returnperiod_hans_

## 0.25° Resolution/ERA5 Analysis/1-day Precip.: Time-Series of Weighted Catchment Daily Precipitation and Return Period Analysis

### Dataset/resolution choices

In [10]:
# %% [Setup & dataset selection]
# ─────────────────────────────────────────────────────────────────────────────
# CHANGE ONLY THE TWO LINES MARKED ← to switch dataset or accumulation window.
# DATASET, RESOLUTION, and WEIGHT_DIR are derived automatically.
# ─────────────────────────────────────────────────────────────────────────────

import sys
from pathlib import Path

HELPER_DIR = Path("/nird/home/lbal/internship_storm_hans/helper")
if str(HELPER_DIR) not in sys.path:
    sys.path.insert(0, str(HELPER_DIR))

import config_paths as cfg

# ── Dataset selection ─────────────────────────────────────────────────────────
# Options:  "era5_0.5"  |  "era5_0.25"  |  "senorge"
DATASET_KEY = "era5_0.25"      # ← CHANGE THIS

# ── Accumulation window ───────────────────────────────────────────────────────
WINDOW_DAYS = 1              # ← CHANGE THIS (1 or 2)

# ── Recompute flag ────────────────────────────────────────────────────────────
FORCE_RECOMPUTE = False

# ── Figure subfolder ─────────────────────────────────────────────────────────
FIG_SUBDIR = "timeseries_return_hans"

# ── Derived automatically — do not edit below ────────────────────────────────
_CONFIG = {
    "era5_0.5":  {"dataset": "era5",    "resolution": "0.5x0.5",   "weight_dir": None},
    "era5_0.25": {"dataset": "era5",    "resolution": "0.25x0.25", "weight_dir": cfg.WEIGHTS_025_DIR},
    "senorge":   {"dataset": "senorge", "resolution": "",           "weight_dir": None},
}
if DATASET_KEY not in _CONFIG:
    raise ValueError(f"Unknown DATASET_KEY '{DATASET_KEY}'. Choose from: {list(_CONFIG)}")

DATASET    = _CONFIG[DATASET_KEY]["dataset"]
RESOLUTION = _CONFIG[DATASET_KEY]["resolution"]
WEIGHT_DIR = _CONFIG[DATASET_KEY]["weight_dir"]

print(f"Dataset    : {DATASET}")
print(f"Resolution : {RESOLUTION or 'n/a (seNorge — one grid only)'}")
print(f"Window     : {WINDOW_DAYS}-day")
print(f"Recompute  : {FORCE_RECOMPUTE}")
print(f"Weight dir : {WEIGHT_DIR or '(default: CATCHMENT_RAW_DIR)'}")
print(f"Fig subdir : {FIG_SUBDIR}")

Dataset    : era5
Resolution : 0.25x0.25
Window     : 1-day
Recompute  : False
Weight dir : /nird/datalake/NS9873K/lbal/postprocessed/0.25_weights
Fig subdir : timeseries_return_hans


### Configuration check

In [11]:
# %% [Configuration check]
import config_paths as cfg

tag = cfg.res_tag(DATASET, RESOLUTION)

print(f"Resolution tag     : {tag}")
print(f"Postproc dir       : {cfg.postproc_dir(DATASET)}")
print(f"Figures (primary)  : {cfg.FIGURES_DIR / FIG_SUBDIR}")
print(f"Figures (secondary): {cfg.FIGURES_DIR_SECONDARY / FIG_SUBDIR}")
print(f"Hans year          : {cfg.HANS_SEARCH_YEAR}")

# ── File discovery — branches on dataset ─────────────────────────────────────
if DATASET == "senorge":
    from data_senorge import find_senorge_files, get_year_range_senorge
    _raw_files = find_senorge_files(cfg.SENORGE_RAW_DIR)
    start_year, end_year = get_year_range_senorge(_raw_files)
    print(f"seNorge raw dir    : {cfg.SENORGE_RAW_DIR}")
else:
    from data_era5 import find_era5_files, get_year_range
    _raw_files = find_era5_files(cfg.ERA5_RAW_DIR, RESOLUTION)
    start_year, end_year = get_year_range(_raw_files, RESOLUTION)
    print(f"ERA5 raw dir       : {cfg.ERA5_RAW_DIR}")

print(f"Files found        : {len(_raw_files)}  ({start_year}–{end_year})")
print()
print("Postprocessed cache status:")
cached_count = 0
missing = []

for slug, title in cfg.CATCHMENTS.items():
    nc = cfg.catchment_postproc_path(
        DATASET, RESOLUTION, WINDOW_DAYS, slug, start_year, end_year
    )
    exists = nc.exists()
    if exists:
        cached_count += 1
    else:
        missing.append(slug)
    print(f"  {slug:30s}  {'✓ cached' if exists else '  (not yet cached)'}")

print()
print(f"Cached files       : {cached_count}/{len(cfg.CATCHMENTS)}")

if FORCE_RECOMPUTE:
    print("Run behavior       : FORCE_RECOMPUTE=True → all catchments will be recomputed from raw data")
elif missing:
    print("Run behavior       : cached catchments will be reused; missing catchments computed from raw data")
else:
    print("Run behavior       : all catchments cached → raw data will not be loaded")

Resolution tag     : era5_0.25x0.25
Postproc dir       : /nird/datalake/NS9873K/lbal/postprocessed/era5
Figures (primary)  : /nird/datalake/NS9873K/lbal/figures/timeseries_return_hans
Figures (secondary): /nird/home/lbal/internship_storm_hans/figures/timeseries_return_hans
Hans year          : 2023
ERA5 raw dir       : /nird/datapeak/NS9873K/etdu/raw/era5/continuous-format/europe/daily/tp24
Files found        : 84  (1941–2024)

Postprocessed cache status:
  nevina_bergheim                 ✓ cached
  nevina_honnefoss                ✓ cached
  nevina_losna                    ✓ cached
  regine_drammen                  ✓ cached
  regine_glomma                   ✓ cached

Cached files       : 5/5
Run behavior       : all catchments cached → raw data will not be loaded


### Run full analysis on all catchments

In [12]:
# %% [Run full analysis]
from catchment_tools import run_all

run_all(
    dataset         = DATASET,
    resolution      = RESOLUTION,
    window_days     = WINDOW_DAYS,
    force_recompute = FORCE_RECOMPUTE,
    fig_subdir      = FIG_SUBDIR,
    weight_dir      = WEIGHT_DIR,
)


[run_all] Dataset: era5 | Resolution: 0.25x0.25
[run_all] Files found: 84  (1941–2024)

── Catchment: Nevina Bergheim (nevina_bergheim) ──
  [cache] Found postprocessed file → post_processed_era5_0.25x0.25_1day_nevina_bergheim_1941-2024.nc
    [fig]   Saved → /nird/datalake/NS9873K/lbal/figures/timeseries_return_hans/timeseries_returnperiod_hans_era5_0.25x0.25_1day_nevina_bergheim_1941-2024.pdf
    [fig]   Saved → /nird/home/lbal/internship_storm_hans/figures/timeseries_return_hans/timeseries_returnperiod_hans_era5_0.25x0.25_1day_nevina_bergheim_1941-2024.pdf

── Catchment: Nevina Hønnefoss (nevina_honnefoss) ──
  [cache] Found postprocessed file → post_processed_era5_0.25x0.25_1day_nevina_honnefoss_1941-2024.nc
    [fig]   Saved → /nird/datalake/NS9873K/lbal/figures/timeseries_return_hans/timeseries_returnperiod_hans_era5_0.25x0.25_1day_nevina_honnefoss_1941-2024.pdf
    [fig]   Saved → /nird/home/lbal/internship_storm_hans/figures/timeseries_return_hans/timeseries_returnperiod_hans_

## 1x1km Resolution/Senorge/2-day Precip.: Time-Series of Weighted Catchment Daily Precipitation and Return Period Analysis

### Dataset/resolution choices

In [13]:
# %% [Setup & dataset selection]
# ─────────────────────────────────────────────────────────────────────────────
# CHANGE ONLY THE TWO LINES MARKED ← to switch dataset or accumulation window.
# DATASET, RESOLUTION, and WEIGHT_DIR are derived automatically.
# ─────────────────────────────────────────────────────────────────────────────

import sys
from pathlib import Path

HELPER_DIR = Path("/nird/home/lbal/internship_storm_hans/helper")
if str(HELPER_DIR) not in sys.path:
    sys.path.insert(0, str(HELPER_DIR))

import config_paths as cfg

# ── Dataset selection ─────────────────────────────────────────────────────────
# Options:  "era5_0.5"  |  "era5_0.25"  |  "senorge"
DATASET_KEY = "senorge"      # ← CHANGE THIS

# ── Accumulation window ───────────────────────────────────────────────────────
WINDOW_DAYS = 2              # ← CHANGE THIS (1 or 2)

# ── Recompute flag ────────────────────────────────────────────────────────────
FORCE_RECOMPUTE = False

# ── Figure subfolder ─────────────────────────────────────────────────────────
FIG_SUBDIR = "timeseries_return_hans"

# ── Derived automatically — do not edit below ────────────────────────────────
_CONFIG = {
    "era5_0.5":  {"dataset": "era5",    "resolution": "0.5x0.5",   "weight_dir": None},
    "era5_0.25": {"dataset": "era5",    "resolution": "0.25x0.25", "weight_dir": cfg.WEIGHTS_025_DIR},
    "senorge":   {"dataset": "senorge", "resolution": "",           "weight_dir": None},
}
if DATASET_KEY not in _CONFIG:
    raise ValueError(f"Unknown DATASET_KEY '{DATASET_KEY}'. Choose from: {list(_CONFIG)}")

DATASET    = _CONFIG[DATASET_KEY]["dataset"]
RESOLUTION = _CONFIG[DATASET_KEY]["resolution"]
WEIGHT_DIR = _CONFIG[DATASET_KEY]["weight_dir"]

print(f"Dataset    : {DATASET}")
print(f"Resolution : {RESOLUTION or 'n/a (seNorge — one grid only)'}")
print(f"Window     : {WINDOW_DAYS}-day")
print(f"Recompute  : {FORCE_RECOMPUTE}")
print(f"Weight dir : {WEIGHT_DIR or '(default: CATCHMENT_RAW_DIR)'}")
print(f"Fig subdir : {FIG_SUBDIR}")

Dataset    : senorge
Resolution : n/a (seNorge — one grid only)
Window     : 2-day
Recompute  : False
Weight dir : (default: CATCHMENT_RAW_DIR)
Fig subdir : timeseries_return_hans


### Configuration check

In [14]:
# %% [Configuration check]
import config_paths as cfg

tag = cfg.res_tag(DATASET, RESOLUTION)

print(f"Resolution tag     : {tag}")
print(f"Postproc dir       : {cfg.postproc_dir(DATASET)}")
print(f"Figures (primary)  : {cfg.FIGURES_DIR / FIG_SUBDIR}")
print(f"Figures (secondary): {cfg.FIGURES_DIR_SECONDARY / FIG_SUBDIR}")
print(f"Hans year          : {cfg.HANS_SEARCH_YEAR}")

# ── File discovery — branches on dataset ─────────────────────────────────────
if DATASET == "senorge":
    from data_senorge import find_senorge_files, get_year_range_senorge
    _raw_files = find_senorge_files(cfg.SENORGE_RAW_DIR)
    start_year, end_year = get_year_range_senorge(_raw_files)
    print(f"seNorge raw dir    : {cfg.SENORGE_RAW_DIR}")
else:
    from data_era5 import find_era5_files, get_year_range
    _raw_files = find_era5_files(cfg.ERA5_RAW_DIR, RESOLUTION)
    start_year, end_year = get_year_range(_raw_files, RESOLUTION)
    print(f"ERA5 raw dir       : {cfg.ERA5_RAW_DIR}")

print(f"Files found        : {len(_raw_files)}  ({start_year}–{end_year})")
print()
print("Postprocessed cache status:")
cached_count = 0
missing = []

for slug, title in cfg.CATCHMENTS.items():
    nc = cfg.catchment_postproc_path(
        DATASET, RESOLUTION, WINDOW_DAYS, slug, start_year, end_year
    )
    exists = nc.exists()
    if exists:
        cached_count += 1
    else:
        missing.append(slug)
    print(f"  {slug:30s}  {'✓ cached' if exists else '  (not yet cached)'}")

print()
print(f"Cached files       : {cached_count}/{len(cfg.CATCHMENTS)}")

if FORCE_RECOMPUTE:
    print("Run behavior       : FORCE_RECOMPUTE=True → all catchments will be recomputed from raw data")
elif missing:
    print("Run behavior       : cached catchments will be reused; missing catchments computed from raw data")
else:
    print("Run behavior       : all catchments cached → raw data will not be loaded")

Resolution tag     : senorge
Postproc dir       : /nird/datalake/NS9873K/lbal/postprocessed/senorge
Figures (primary)  : /nird/datalake/NS9873K/lbal/figures/timeseries_return_hans
Figures (secondary): /nird/home/lbal/internship_storm_hans/figures/timeseries_return_hans
Hans year          : 2023
seNorge raw dir    : /nird/datapeak/NS9873K/DATA/senorge/rr
Files found        : 69  (1957–2025)

Postprocessed cache status:
  nevina_bergheim                 ✓ cached
  nevina_honnefoss                ✓ cached
  nevina_losna                    ✓ cached
  regine_drammen                  ✓ cached
  regine_glomma                   ✓ cached

Cached files       : 5/5
Run behavior       : all catchments cached → raw data will not be loaded


### Run full analysis on all catchments

In [15]:
# %% [Run full analysis]
from catchment_tools import run_all

run_all(
    dataset         = DATASET,
    resolution      = RESOLUTION,
    window_days     = WINDOW_DAYS,
    force_recompute = FORCE_RECOMPUTE,
    fig_subdir      = FIG_SUBDIR,
    weight_dir      = WEIGHT_DIR,
)


[run_all] Dataset: senorge | Resolution: n/a
[run_all] Files found: 69  (1957–2025)

── Catchment: Nevina Bergheim (nevina_bergheim) ──
  [cache] Found postprocessed file → post_processed_senorge_2day_nevina_bergheim_1957-2025.nc
    [fig]   Saved → /nird/datalake/NS9873K/lbal/figures/timeseries_return_hans/timeseries_returnperiod_hans_senorge_2day_nevina_bergheim_1957-2025.pdf
    [fig]   Saved → /nird/home/lbal/internship_storm_hans/figures/timeseries_return_hans/timeseries_returnperiod_hans_senorge_2day_nevina_bergheim_1957-2025.pdf

── Catchment: Nevina Hønnefoss (nevina_honnefoss) ──
  [cache] Found postprocessed file → post_processed_senorge_2day_nevina_honnefoss_1957-2025.nc
    [fig]   Saved → /nird/datalake/NS9873K/lbal/figures/timeseries_return_hans/timeseries_returnperiod_hans_senorge_2day_nevina_honnefoss_1957-2025.pdf
    [fig]   Saved → /nird/home/lbal/internship_storm_hans/figures/timeseries_return_hans/timeseries_returnperiod_hans_senorge_2day_nevina_honnefoss_1957-202

## 1x1km Resolution/Senorge/1-day Precip.: Time-Series of Weighted Catchment Daily Precipitation and Return Period Analysis

### Dataset/resolution choices

In [16]:
# %% [Setup & dataset selection]
# ─────────────────────────────────────────────────────────────────────────────
# CHANGE ONLY THE TWO LINES MARKED ← to switch dataset or accumulation window.
# DATASET, RESOLUTION, and WEIGHT_DIR are derived automatically.
# ─────────────────────────────────────────────────────────────────────────────

import sys
from pathlib import Path

HELPER_DIR = Path("/nird/home/lbal/internship_storm_hans/helper")
if str(HELPER_DIR) not in sys.path:
    sys.path.insert(0, str(HELPER_DIR))

import config_paths as cfg

# ── Dataset selection ─────────────────────────────────────────────────────────
# Options:  "era5_0.5"  |  "era5_0.25"  |  "senorge"
DATASET_KEY = "senorge"      # ← CHANGE THIS

# ── Accumulation window ───────────────────────────────────────────────────────
WINDOW_DAYS = 1              # ← CHANGE THIS (1 or 2)

# ── Recompute flag ────────────────────────────────────────────────────────────
FORCE_RECOMPUTE = False

# ── Figure subfolder ─────────────────────────────────────────────────────────
FIG_SUBDIR = "timeseries_return_hans"

# ── Derived automatically — do not edit below ────────────────────────────────
_CONFIG = {
    "era5_0.5":  {"dataset": "era5",    "resolution": "0.5x0.5",   "weight_dir": None},
    "era5_0.25": {"dataset": "era5",    "resolution": "0.25x0.25", "weight_dir": cfg.WEIGHTS_025_DIR},
    "senorge":   {"dataset": "senorge", "resolution": "",           "weight_dir": None},
}
if DATASET_KEY not in _CONFIG:
    raise ValueError(f"Unknown DATASET_KEY '{DATASET_KEY}'. Choose from: {list(_CONFIG)}")

DATASET    = _CONFIG[DATASET_KEY]["dataset"]
RESOLUTION = _CONFIG[DATASET_KEY]["resolution"]
WEIGHT_DIR = _CONFIG[DATASET_KEY]["weight_dir"]

print(f"Dataset    : {DATASET}")
print(f"Resolution : {RESOLUTION or 'n/a (seNorge — one grid only)'}")
print(f"Window     : {WINDOW_DAYS}-day")
print(f"Recompute  : {FORCE_RECOMPUTE}")
print(f"Weight dir : {WEIGHT_DIR or '(default: CATCHMENT_RAW_DIR)'}")
print(f"Fig subdir : {FIG_SUBDIR}")

Dataset    : senorge
Resolution : n/a (seNorge — one grid only)
Window     : 1-day
Recompute  : False
Weight dir : (default: CATCHMENT_RAW_DIR)
Fig subdir : timeseries_return_hans


### Configuration check

In [17]:
# %% [Configuration check]
import config_paths as cfg

tag = cfg.res_tag(DATASET, RESOLUTION)

print(f"Resolution tag     : {tag}")
print(f"Postproc dir       : {cfg.postproc_dir(DATASET)}")
print(f"Figures (primary)  : {cfg.FIGURES_DIR / FIG_SUBDIR}")
print(f"Figures (secondary): {cfg.FIGURES_DIR_SECONDARY / FIG_SUBDIR}")
print(f"Hans year          : {cfg.HANS_SEARCH_YEAR}")

# ── File discovery — branches on dataset ─────────────────────────────────────
if DATASET == "senorge":
    from data_senorge import find_senorge_files, get_year_range_senorge
    _raw_files = find_senorge_files(cfg.SENORGE_RAW_DIR)
    start_year, end_year = get_year_range_senorge(_raw_files)
    print(f"seNorge raw dir    : {cfg.SENORGE_RAW_DIR}")
else:
    from data_era5 import find_era5_files, get_year_range
    _raw_files = find_era5_files(cfg.ERA5_RAW_DIR, RESOLUTION)
    start_year, end_year = get_year_range(_raw_files, RESOLUTION)
    print(f"ERA5 raw dir       : {cfg.ERA5_RAW_DIR}")

print(f"Files found        : {len(_raw_files)}  ({start_year}–{end_year})")
print()
print("Postprocessed cache status:")
cached_count = 0
missing = []

for slug, title in cfg.CATCHMENTS.items():
    nc = cfg.catchment_postproc_path(
        DATASET, RESOLUTION, WINDOW_DAYS, slug, start_year, end_year
    )
    exists = nc.exists()
    if exists:
        cached_count += 1
    else:
        missing.append(slug)
    print(f"  {slug:30s}  {'✓ cached' if exists else '  (not yet cached)'}")

print()
print(f"Cached files       : {cached_count}/{len(cfg.CATCHMENTS)}")

if FORCE_RECOMPUTE:
    print("Run behavior       : FORCE_RECOMPUTE=True → all catchments will be recomputed from raw data")
elif missing:
    print("Run behavior       : cached catchments will be reused; missing catchments computed from raw data")
else:
    print("Run behavior       : all catchments cached → raw data will not be loaded")

Resolution tag     : senorge
Postproc dir       : /nird/datalake/NS9873K/lbal/postprocessed/senorge
Figures (primary)  : /nird/datalake/NS9873K/lbal/figures/timeseries_return_hans
Figures (secondary): /nird/home/lbal/internship_storm_hans/figures/timeseries_return_hans
Hans year          : 2023
seNorge raw dir    : /nird/datapeak/NS9873K/DATA/senorge/rr
Files found        : 69  (1957–2025)

Postprocessed cache status:
  nevina_bergheim                   (not yet cached)
  nevina_honnefoss                  (not yet cached)
  nevina_losna                      (not yet cached)
  regine_drammen                    (not yet cached)
  regine_glomma                     (not yet cached)

Cached files       : 0/5
Run behavior       : cached catchments will be reused; missing catchments computed from raw data


### Run full analysis on all catchments

In [18]:
# %% [Run full analysis]
from catchment_tools import run_all

run_all(
    dataset         = DATASET,
    resolution      = RESOLUTION,
    window_days     = WINDOW_DAYS,
    force_recompute = FORCE_RECOMPUTE,
    fig_subdir      = FIG_SUBDIR,
    weight_dir      = WEIGHT_DIR,
)


[run_all] Dataset: senorge | Resolution: n/a
[run_all] Files found: 69  (1957–2025)

── Catchment: Nevina Bergheim (nevina_bergheim) ──
  [raw] Loading raw data because at least one catchment needs recomputation ...
  Opening 69 seNorge files (lazy) ...


/nird/home/lbal/internship_storm_hans/helper/data_senorge.py:57: UserWarning: The specified chunks separate the stored chunks along dimension "Y" starting at index 256. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_mfdataset(
/nird/home/lbal/internship_storm_hans/helper/data_senorge.py:57: UserWarning: The specified chunks separate the stored chunks along dimension "X" starting at index 256. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_mfdataset(
/nird/home/lbal/internship_storm_hans/helper/data_senorge.py:57: UserWarning: The specified chunks separate the stored chunks along dimension "Y" starting at index 256. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_mfdataset(
/nird/home/lbal/internship_storm_hans/helper/data_senorge.py:57: UserWarning: The specified chunks separate the stored chunks along dimension "X" starting at index 256. This could degrade

  Computing weighted mean (lazy over time chunks) ...
    [cache] Saved → post_processed_senorge_1day_nevina_bergheim_1957-2025.nc
    [fig]   Saved → /nird/datalake/NS9873K/lbal/figures/timeseries_return_hans/timeseries_returnperiod_hans_senorge_1day_nevina_bergheim_1957-2025.pdf
    [fig]   Saved → /nird/home/lbal/internship_storm_hans/figures/timeseries_return_hans/timeseries_returnperiod_hans_senorge_1day_nevina_bergheim_1957-2025.pdf

── Catchment: Nevina Hønnefoss (nevina_honnefoss) ──
  Computing weighted mean (lazy over time chunks) ...
    [cache] Saved → post_processed_senorge_1day_nevina_honnefoss_1957-2025.nc
    [fig]   Saved → /nird/datalake/NS9873K/lbal/figures/timeseries_return_hans/timeseries_returnperiod_hans_senorge_1day_nevina_honnefoss_1957-2025.pdf
    [fig]   Saved → /nird/home/lbal/internship_storm_hans/figures/timeseries_return_hans/timeseries_returnperiod_hans_senorge_1day_nevina_honnefoss_1957-2025.pdf

── Catchment: Nevina Losna (nevina_losna) ──
  Computing